In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mplconfig")

import numpy as np
import yt
import logging

yt.set_log_level("ERROR")
logging.getLogger("yt").setLevel(logging.ERROR)

plotfile = sorted(Path("diags").glob("plt*"))[-1]
ds = yt.load(str(plotfile))

grid = ds.covering_grid(
    level=0,
    left_edge=ds.domain_left_edge,
    dims=ds.domain_dimensions,
)

Ex = grid[("boxlib", "Ex")].to_ndarray()
Ey = grid[("boxlib", "Ey")].to_ndarray()
Ez = grid[("boxlib", "Ez")].to_ndarray()
Bx = grid[("boxlib", "Bx")].to_ndarray()
By = grid[("boxlib", "By")].to_ndarray()
Bz = grid[("boxlib", "Bz")].to_ndarray()

fields = {
    "Ex": Ex,
    "Ey": Ey,
    "Ez": Ez,
    "|E|": np.sqrt(Ex**2 + Ey**2 + Ez**2),
    "Bx": Bx,
    "By": By,
    "Bz": Bz,
    "|B|": np.sqrt(Bx**2 + By**2 + Bz**2),
}

print(f"Loaded {plotfile}")
print(f"Available fields: {', '.join(fields)}")
print(f"Grid shape: {Ex.shape}  # x, y, z")
print(f"|E| min/max: {fields['|E|'].min():.6g}, {fields['|E|'].max():.6g}")

In [ ]:
import matplotlib.pyplot as plt

# --- selection ---
variable = "|E|"  # Ex, Ey, Ez, |E|, Bx, By, Bz, |B|
direction = "z"   # x, y, z
# -----------------

dir_index = {"x": 0, "y": 1, "z": 2}[direction]
dims = fields[variable].shape
centers = [n // 2 for n in dims]

edges = np.linspace(
    float(ds.domain_left_edge[dir_index]),
    float(ds.domain_right_edge[dir_index]),
    dims[dir_index] + 1,
)
coord = 0.5 * (edges[:-1] + edges[1:])

idx = list(centers)
idx[dir_index] = slice(None)
fixed = {axis: centers[i] for i, axis in enumerate("xyz") if i != dir_index}
line = fields[variable][tuple(idx)]

plt.figure(figsize=(7, 4))
plt.plot(coord * 1e6, line, color="black", linewidth=2)
plt.xlabel(f"{direction} (um)")
plt.ylabel(variable)
plt.title(
    f"{variable} along {direction} at "
    + ", ".join(f"{axis}-index {i}" for axis, i in fixed.items())
)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation, FFMpegWriter
from tqdm import tqdm

# --- selection ---
variable = "|E|"  # Ex, Ey, Ez, |E|, Bx, By, Bz, |B|
direction = "z"   # x, y, z
# -----------------

yt.set_log_level("ERROR")
logging.getLogger("yt").setLevel(logging.ERROR)

diag_dir = Path("diags")
plotfiles = sorted(
    diag_dir.glob("plt*"),
    key=lambda path: int(path.name.removeprefix("plt")),
)

stride = 1
frames = plotfiles[::stride]
dir_index = {"x": 0, "y": 1, "z": 2}[direction]


def lineout(path):
    frame_ds = yt.load(str(path))
    frame_grid = frame_ds.covering_grid(
        level=0,
        left_edge=frame_ds.domain_left_edge,
        dims=frame_ds.domain_dimensions,
    )
    quantities = {
        "Ex": frame_grid[("boxlib", "Ex")].to_ndarray(),
        "Ey": frame_grid[("boxlib", "Ey")].to_ndarray(),
        "Ez": frame_grid[("boxlib", "Ez")].to_ndarray(),
        "Bx": frame_grid[("boxlib", "Bx")].to_ndarray(),
        "By": frame_grid[("boxlib", "By")].to_ndarray(),
        "Bz": frame_grid[("boxlib", "Bz")].to_ndarray(),
    }
    quantities["|E|"] = np.sqrt(
        quantities["Ex"] ** 2 + quantities["Ey"] ** 2 + quantities["Ez"] ** 2
    )
    quantities["|B|"] = np.sqrt(
        quantities["Bx"] ** 2 + quantities["By"] ** 2 + quantities["Bz"] ** 2
    )

    field = quantities[variable]
    dims = field.shape
    centers = [n // 2 for n in dims]
    edges = np.linspace(
        float(frame_ds.domain_left_edge[dir_index]),
        float(frame_ds.domain_right_edge[dir_index]),
        dims[dir_index] + 1,
    )
    coord = 0.5 * (edges[:-1] + edges[1:])
    idx = list(centers)
    idx[dir_index] = slice(None)
    return float(frame_ds.current_time), coord, field[tuple(idx)]


times = []
lines = []
for path in tqdm(frames):
    time, coord, line_values = lineout(path)
    times.append(time)
    lines.append(line_values)

times = np.asarray(times)
lines = np.asarray(lines)

fig, ax = plt.subplots(figsize=(7, 4))
(line,) = ax.plot(coord * 1e6, lines[0], color="black", linewidth=2)
title = ax.set_title(f"{variable} along {direction}, t = {times[0]:.3e} s")
ax.set_xlabel(f"{direction} (um)")
ax.set_ylabel(variable)
ax.set_xlim(float(coord[0] * 1e6), float(coord[-1] * 1e6))
ymin = float(min(0.0, lines.min())) if variable.startswith("|") else float(lines.min())
ymax = float(lines.max())
pad = 0.05 * max(abs(ymax - ymin), abs(ymax), 1.0)
ax.set_ylim(ymin - (0.0 if variable.startswith("|") else pad), ymax + pad)
ax.grid(True, alpha=0.3)


def update(frame_index):
    line.set_ydata(lines[frame_index])
    title.set_text(f"{variable} along {direction}, t = {times[frame_index]:.3e} s")
    return line, title


animation = FuncAnimation(fig, update, frames=len(lines), interval=50, blit=False)

safe_var = variable.replace("|", "").replace(" ", "")
video_path = Path(f"wave_freespace_run/{safe_var}_along_{direction}.webm")
video_path.parent.mkdir(parents=True, exist_ok=True)
writer = FFMpegWriter(fps=20, codec="libvpx-vp9", bitrate=1800)
animation.save(video_path, writer=writer, dpi=150)
plt.close(fig)

print(f"Wrote {video_path}")
print(f"Frames: {len(lines)}")